In [1]:
import os
os.chdir("../../..")
print(os.getcwd())

/Users/titonka/FAIRIS


In [2]:
from sklearn.mixture import GaussianMixture
import numpy as np
from sklearn.cluster import KMeans, Birch
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import math
import pickle
import logging
import time
from PIL import Image

In [3]:
def _load_bg_image(image_path, flip_vertical=True):
    arr = np.asarray(Image.open(image_path).convert("RGB"))
    if flip_vertical:
        arr = np.flipud(arr)  # so +Y is up
    return arr

In [4]:
# Function to plot clusters on the x, y plane
def plot_clusters(xy_list, cluster_labels, name):
    """
    Plot the clusters on the x, y plane using the original (x, y) coordinates, and save the figure.

    Args:
    - xy_list (list of tuples): The original x, y coordinates for each datapoint.
    - cluster_labels (list of int): The cluster label for each datapoint.
    - name (str): The filename to save the figure as (e.g., "clusters_plot.png").
    """
    # Convert xy_list to NumPy arrays for easy plotting
    x_coords = np.array([x for x, y in xy_list])
    y_coords = np.array([y for x, y in xy_list])

    # Check if the lengths match
    if len(x_coords) != len(cluster_labels):
        raise ValueError(f"Mismatch: {len(x_coords)} coordinates and {len(cluster_labels)} cluster labels.")

    # Scatter plot with color coding for clusters
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(x_coords, y_coords, c=cluster_labels, cmap='rainbow', alpha=0.7)

    # Add color bar to indicate clusters
    plt.colorbar(scatter, label='Cluster')

    # Add labels and title
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.title(f'Clustering with {len(set(cluster_labels))} Clusters on X-Y Plane')

    # Save the plot to the specified file
    plt.savefig(name)

    # Close the plot to avoid displaying it when running in scripts
    plt.close()


def plot_clusters_by_subplots(xy_list, 
                              theta_list,
                              cluster_labels,
                              name,
                              n_clusters=8,
                              width = 6,
                              height = 6,
                              image_path=None,
                              world_width_m=None,
                              world_height_m=None,
                              image_alpha=0.55,
                              flip_image_vertical=True):
    """
    Plot the clusters on subplots, one for each cluster, using the original (x, y) coordinates
    and their corresponding direction vectors.

    Args:
    - xy_list (list of tuples): The original x, y coordinates for each datapoint.
    - theta_list (list of floats): The orientation (theta) in degrees for each datapoint.
    - cluster_labels (list of int): The cluster label for each datapoint.
    - name (str): The filename to save the plot.
    - n_clusters (int): Number of clusters to plot.
    """
    # Fixed number of columns
    cols = 5
    # Calculate the number of rows needed
    rows = int(math.ceil(n_clusters / cols))

    # Dynamically scale the figsize based on rows and columns
    width_per_col = 6  # Adjust this for horizontal scaling
    height_per_row = 6  # Adjust this for vertical scaling
    figsize = (cols * width_per_col, rows * height_per_row)

    # Convert xy_list to NumPy arrays for easy filtering and plotting
    x_coords = np.array([x for x, y in xy_list])
    y_coords = np.array([y for x, y in xy_list])
    theta_list = np.array([theta for theta in theta_list])
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()  # Flatten the axes array for easy indexing
    
    # World dimensions and axis limits
    if (world_width_m is not None) and (world_height_m is not None):
        xlim = (-world_width_m / 2.0,  world_width_m / 2.0)
        ylim = (-world_height_m / 2.0, world_height_m / 2.0)
        extent = [xlim[0], xlim[1], ylim[0], ylim[1]]
    else:
        # fallback to legacy symmetric limits
        xlim = (-width / 2.0,  width / 2.0)
        ylim = (-height / 2.0, height / 2.0)
        extent = [xlim[0], xlim[1], ylim[0], ylim[1]]

    # Load background once (if provided)
    bg = None
    if image_path is not None:
        bg = _load_bg_image(image_path, flip_vertical=flip_image_vertical)

    for cluster_id in range(n_clusters):
        if bg is not None:
            axes[cluster_id].imshow(bg, extent=extent, origin='lower', alpha=image_alpha, zorder=0)
        # Get the indices of the datapoints that belong to the current cluster
        cluster_indices = np.where(cluster_labels == cluster_id)[0]

        # Filter x and y coordinates for the current cluster
        cluster_x = x_coords[cluster_indices]
        cluster_y = y_coords[cluster_indices]

        # Extract theta values for the current cluster
        cluster_theta = theta_list[cluster_indices]

        # Compute dx and dy for each point in the current cluster
        cluster_dx = 0.5 * np.cos(np.radians(cluster_theta))
        cluster_dy = 0.5 * np.sin(np.radians(cluster_theta))

        # Scatter plot for the current cluster
        axes[cluster_id].scatter(cluster_x, cluster_y, c=f'C{cluster_id}', alpha=0.7, label='Points')

        # Add quiver plot for vectors
        # axes[cluster_id].quiver(cluster_x, cluster_y, cluster_dx, cluster_dy, angles='xy', scale_units='xy', scale=1,
        #                         color='black', alpha=0.7, label='Vectors')

        # Set subplot title and labels
        axes[cluster_id].set_title(f'Cluster {cluster_id}')
        axes[cluster_id].set_xlabel('X Coordinate')
        axes[cluster_id].set_ylabel('Y Coordinate')
        axes[cluster_id].set_xlim(-width/2, width/2)  # Set x-axis limits
        axes[cluster_id].set_ylim(-height/2, width/2)  # Set y-axis limits
        axes[cluster_id].legend()

    # Hide unused subplots (if n_clusters < len(axes))
    for i in range(n_clusters, len(axes)):
        axes[i].axis('off')

    # Adjust layout for better spacing
    plt.tight_layout()

    # Save the plot to the specified file
    fig.savefig(name)
    plt.close()

def format_data_for_clustering(data):
    multimodal_feature_vectors = []
    cnn_feature_vectors = []
    xy_list = []
    theta_list = []
    for observation in data.observations:
        multimodal_feature_vectors.append(observation.multimodal_feature_vector)
        cnn_feature_vectors.append(observation.cnn_feature_vector)
        xy_list.append((observation.x, observation.y))
        theta_list.append(observation.theta)

    return multimodal_feature_vectors, cnn_feature_vectors, xy_list, theta_list


def cluster_with_kmeans_and_save_centers(features_list, n_clusters, centers_save_path):
    """
    Perform KMeans clustering, calculate the maximum distance for each cluster,
    and save the cluster centers and max distances as a list of lists using pickle.

    Args:
    - features_list (list of numpy arrays): The feature vectors extracted from the images.
    - n_clusters (int): The number of clusters to form.
    - centers_save_path (str): Path to save the cluster centers and max distances using pickle.

    Returns:
    - cluster_labels (list of int): The cluster label for each datapoint.
    - cluster_centers (numpy array): The centers of the final clusters.
    """
    features_array = np.array(features_list)

    # Perform KMeans clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(features_array)

    # Get cluster centers and labels for each data point
    cluster_centers = kmeans.cluster_centers_
    labels = kmeans.labels_

    # List to store [center, max_distance] for each cluster
    cluster_data = []

    # Calculate max distance for each cluster
    for cluster_index in range(n_clusters):
        # Get data points belonging to this cluster
        cluster_points = features_array[labels == cluster_index]

        # Calculate distances from each point to the cluster center
        distances = cdist(cluster_points, [cluster_centers[cluster_index]], metric='euclidean').flatten()

        # Find the maximum distance for this cluster
        max_distance = distances.max()

        # Append the center and max distance as a pair to cluster_data
        cluster_data.append([cluster_centers[cluster_index].tolist(), max_distance])

    # Save cluster_data (centers and max distances) using pickle
    with open(centers_save_path, 'wb') as f:
        pickle.dump(cluster_data, f)

    return cluster_labels

In [5]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def cluster_with_gmm_and_save_centers(features_list, n_clusters, centers_save_path, use_pca=False, n_components=100):
    """
    Perform Gaussian Mixture Model (GMM) clustering, calculate the maximum distance for each cluster,
    and save the cluster means and max distances as a list of lists using pickle.
    Optionally apply PCA to reduce dimensionality.

    Args:
        features_list (list of numpy arrays): The feature vectors extracted from the images.
        n_clusters (int): The number of clusters to form.
        centers_save_path (str): Path to save the cluster means and max distances using pickle.
        use_pca (bool): Whether to apply PCA for dimensionality reduction. Default is False.
        n_components (int): Number of components to keep if PCA is used. Default is 100.

    Returns:
        cluster_labels (list of int): The cluster label for each datapoint (based on highest probability).
    """
    start_time = time.time()
    logging.info("Starting GMM clustering with %d data points and %d clusters", len(features_list), n_clusters)

    # Convert features_list to numpy array
    try:
        features_array = np.array(features_list, dtype=np.float32)
        if np.any(np.isnan(features_array)) or np.any(np.isinf(features_array)):
            raise ValueError("Feature array contains NaN or infinite values")
    except Exception as e:
        logging.error("Error converting features_list to array: %s", e)
        raise

    # Log feature dimensions
    logging.info("Feature array shape: %s", features_array.shape)

    # Apply PCA if enabled
    if use_pca:
        logging.info("Applying PCA to reduce dimensionality to %d components", n_components)
        try:
            pca = PCA(n_components=n_components)
            features_array = pca.fit_transform(features_array)
            logging.info("PCA reduced shape: %s", features_array.shape)
        except Exception as e:
            logging.error("Error during PCA: %s", e)
            raise

    # Perform GMM clustering
    try:
        gmm = GaussianMixture(
            n_components=n_clusters,
            covariance_type='diag',  # Corrected to 'diag'
            max_iter=50,
            random_state=42,
            verbose=1,
            verbose_interval=10
        )
        gmm.fit(features_array)
        cluster_labels = gmm.predict(features_array)
        logging.info("GMM fitting completed in %.2f seconds", time.time() - start_time)
    except Exception as e:
        logging.error("Error during GMM fitting: %s", e)
        raise

    # Get cluster means (equivalent to centroids)
    cluster_means = gmm.means_

    # List to store [mean, max_distance] for each cluster
    cluster_data = []

    # Calculate max distance for each cluster
    for cluster_index in range(n_clusters):
        # Get data points assigned to this cluster
        cluster_points = features_array[cluster_labels == cluster_index]

        # Calculate distances from each point to the cluster mean
        try:
            distances = cdist(cluster_points, [cluster_means[cluster_index]], metric='euclidean').flatten()
            max_distance = distances.max() if len(distances) > 0 else 0.0
        except Exception as e:
            logging.error("Error calculating distances for cluster %d: %s", cluster_index, e)
            max_distance = 0.0

        # Append the mean and max distance as a pair to cluster_data
        cluster_data.append([cluster_means[cluster_index].tolist(), max_distance])
        logging.info("Cluster %d: %d points, max distance %.2f", cluster_index, len(cluster_points), max_distance)

    # Save cluster_data (means and max distances) using pickle
    try:
        with open(centers_save_path, 'wb') as f:
            pickle.dump(cluster_data, f)
        logging.info("Cluster data saved to %s in %.2f seconds", centers_save_path, time.time() - start_time)
    except Exception as e:
        logging.error("Error saving cluster data: %s", e)
        raise

    return cluster_labels

In [6]:
data_dir = 'data/VisualPlaceCellData/'
maze_files = ['LM4','LM6','LM8', 'LM8D','LMO8','LMO8D','LM8_addition','LMO8_remove','BR']
maze_index = 4

with open(data_dir+maze_files[maze_index]+'_Training','rb') as file:
    visual_place_cell_data = pickle.load(file)

multimodal_feature_vectors,cnn_feature_vectors,xy_list,theta_list = format_data_for_clustering(visual_place_cell_data)
n_clusters = [10,25,50,75,100,250,500,750]
for n_cluster in n_clusters:

    centers_save_path = "data/VisualPlaceCellData/VisualPlaceCellClusters/multimodal_gmm_"+str(n_cluster)+"_clusters_"+maze_files[maze_index]
    cluster_labels = cluster_with_gmm_and_save_centers(multimodal_feature_vectors, n_cluster, centers_save_path)

    # Now plot the clusters
    if maze_files[maze_index] == 'BR':
        plot_clusters_by_subplots(xy_list, 
                                  theta_list, 
                                  cluster_labels, "data/figures/Clustering/multimodel_gmm_"+str(n_cluster)+"_"+maze_files[maze_index]+"_clusters.png", 
                                  n_clusters=n_cluster,
                                  width=12.86,
                                  height=7.7,
                                  world_width_m=12.86,
                                  world_height_m=7.7 ,
                                  image_path="data/DataCache/"+maze_files[maze_index]+".png")
    else:
        plot_clusters_by_subplots(xy_list, 
                                  theta_list, 
                                  cluster_labels, "data/figures/Clustering/multimodel_gmm_"+str(n_cluster)+"_"+maze_files[maze_index]+"_clusters.png", 
                                  n_clusters=n_cluster,
                                  width=6,
                                  height=6,
                                  world_width_m=6,
                                  world_height_m=6 ,
                                  image_path="data/DataCache/"+maze_files[maze_index]+".png")


2025-11-20 06:50:00,496 - INFO - Starting GMM clustering with 3772 data points and 10 clusters
2025-11-20 06:50:01,231 - INFO - Feature array shape: (3772, 58912)


Initialization 0
  Iteration 10
  Iteration 20
Initialization converged.


2025-11-20 06:52:20,954 - INFO - GMM fitting completed in 140.46 seconds
2025-11-20 06:52:21,142 - INFO - Cluster 0: 458 points, max distance 6814.57
2025-11-20 06:52:21,209 - INFO - Cluster 1: 287 points, max distance 5963.06
2025-11-20 06:52:21,297 - INFO - Cluster 2: 401 points, max distance 6627.47
2025-11-20 06:52:21,425 - INFO - Cluster 3: 434 points, max distance 7809.68
2025-11-20 06:52:21,509 - INFO - Cluster 4: 377 points, max distance 6114.60
2025-11-20 06:52:21,639 - INFO - Cluster 5: 438 points, max distance 6989.71
2025-11-20 06:52:21,703 - INFO - Cluster 6: 274 points, max distance 6656.95
2025-11-20 06:52:21,897 - INFO - Cluster 7: 523 points, max distance 7093.26
2025-11-20 06:52:21,948 - INFO - Cluster 8: 200 points, max distance 6934.19
2025-11-20 06:52:22,032 - INFO - Cluster 9: 380 points, max distance 7031.76
2025-11-20 06:52:22,047 - INFO - Cluster data saved to data/VisualPlaceCellData/VisualPlaceCellClusters/multimodal_gmm_10_clusters_LMO8 in 141.55 seconds
202

Initialization 0
  Iteration 10
Initialization converged.


2025-11-20 06:54:11,159 - INFO - GMM fitting completed in 107.74 seconds
2025-11-20 06:54:11,204 - INFO - Cluster 0: 140 points, max distance 6385.80
2025-11-20 06:54:11,252 - INFO - Cluster 1: 147 points, max distance 5622.78
2025-11-20 06:54:11,305 - INFO - Cluster 2: 165 points, max distance 6495.35
2025-11-20 06:54:11,358 - INFO - Cluster 3: 168 points, max distance 6274.68
2025-11-20 06:54:11,420 - INFO - Cluster 4: 272 points, max distance 7329.39
2025-11-20 06:54:11,479 - INFO - Cluster 5: 241 points, max distance 6908.10
2025-11-20 06:54:11,539 - INFO - Cluster 6: 188 points, max distance 6359.89
2025-11-20 06:54:11,571 - INFO - Cluster 7: 130 points, max distance 5463.73
2025-11-20 06:54:11,595 - INFO - Cluster 8: 95 points, max distance 5877.93
2025-11-20 06:54:11,628 - INFO - Cluster 9: 140 points, max distance 5423.16
2025-11-20 06:54:11,658 - INFO - Cluster 10: 126 points, max distance 6323.10
2025-11-20 06:54:11,687 - INFO - Cluster 11: 124 points, max distance 5904.58
20

...

2025-11-20 08:22:01,661 - INFO - GMM fitting completed in 4980.29 seconds
2025-11-20 08:22:01,667 - INFO - Cluster 0: 20 points, max distance 4837.80
2025-11-20 08:22:01,672 - INFO - Cluster 1: 17 points, max distance 4009.53
2025-11-20 08:22:01,677 - INFO - Cluster 2: 14 points, max distance 3669.26
2025-11-20 08:22:01,684 - INFO - Cluster 3: 24 points, max distance 3740.91
2025-11-20 08:22:01,690 - INFO - Cluster 4: 21 points, max distance 4243.13
2025-11-20 08:22:01,698 - INFO - Cluster 5: 25 points, max distance 3996.30
2025-11-20 08:22:01,704 - INFO - Cluster 6: 19 points, max distance 3911.07
2025-11-20 08:22:01,711 - INFO - Cluster 7: 24 points, max distance 3479.49
2025-11-20 08:22:01,718 - INFO - Cluster 8: 23 points, max distance 3986.54
2025-11-20 08:22:01,724 - INFO - Cluster 9: 23 points, max distance 4349.67
2025-11-20 08:22:01,729 - INFO - Cluster 10: 15 points, max distance 3904.25
2025-11-20 08:22:01,734 - INFO - Cluster 11: 14 points, max distance 4219.06
2025-11-20 0

Initialization 0
Initialization converged.


2025-11-20 10:29:06,101 - INFO - GMM fitting completed in 5622.52 seconds
2025-11-20 10:29:06,105 - INFO - Cluster 0: 6 points, max distance 3877.56
2025-11-20 10:29:06,109 - INFO - Cluster 1: 10 points, max distance 3518.24
2025-11-20 10:29:06,114 - INFO - Cluster 2: 8 points, max distance 3314.36
2025-11-20 10:29:06,118 - INFO - Cluster 3: 10 points, max distance 3184.52
2025-11-20 10:29:06,122 - INFO - Cluster 4: 7 points, max distance 3098.10
2025-11-20 10:29:06,130 - INFO - Cluster 5: 17 points, max distance 3583.32
2025-11-20 10:29:06,134 - INFO - Cluster 6: 3 points, max distance 2651.59
2025-11-20 10:29:06,138 - INFO - Cluster 7: 10 points, max distance 3265.26
2025-11-20 10:29:06,141 - INFO - Cluster 8: 7 points, max distance 3657.44
2025-11-20 10:29:06,145 - INFO - Cluster 9: 10 points, max distance 3655.49
2025-11-20 10:29:06,148 - INFO - Cluster 10: 6 points, max distance 3541.90
2025-11-20 10:29:06,152 - INFO - Cluster 11: 9 points, max distance 3627.18
2025-11-20 10:29:06

Initialization 0
Initialization converged.


2025-11-20 10:38:40,258 - INFO - GMM fitting completed in 505.13 seconds
2025-11-20 10:38:40,261 - INFO - Cluster 0: 5 points, max distance 3048.09
2025-11-20 10:38:40,264 - INFO - Cluster 1: 6 points, max distance 2852.53
2025-11-20 10:38:40,267 - INFO - Cluster 2: 5 points, max distance 2882.67
2025-11-20 10:38:40,270 - INFO - Cluster 3: 5 points, max distance 2441.09
2025-11-20 10:38:40,273 - INFO - Cluster 4: 5 points, max distance 2782.18
2025-11-20 10:38:40,277 - INFO - Cluster 5: 10 points, max distance 3068.26
2025-11-20 10:38:40,280 - INFO - Cluster 6: 3 points, max distance 2651.59
2025-11-20 10:38:40,283 - INFO - Cluster 7: 7 points, max distance 2876.29
2025-11-20 10:38:40,286 - INFO - Cluster 8: 7 points, max distance 3657.44
2025-11-20 10:38:40,289 - INFO - Cluster 9: 7 points, max distance 3461.44
2025-11-20 10:38:40,292 - INFO - Cluster 10: 5 points, max distance 3294.10
2025-11-20 10:38:40,295 - INFO - Cluster 11: 6 points, max distance 2881.54
2025-11-20 10:38:40,299 